<a href="https://colab.research.google.com/github/vadim1334/---Python/blob/main/HR_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D1%82%D0%B8%D0%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import glob
from openai import OpenAI
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
#библиотеки

In [ ]:
#загрузка
folder = '/content/horeca_recruitment_analytics_28.01.2026-26.02.2026.xlsx'  #или путь к папке с файлами
for filepath in glob.glob(folder + '*.csv') + glob.glob(folder + '*.xlsx'):
    df = load_file(filepath)
    print(f"Загружено: {filepath}, строк: {len(df)}")
    # дальнейшая обработка
    display(df.head())

In [ ]:
#API нейронки
client = OpenAI(
    api_key="",
    base_url="https://api.polza.ai/api/v1"
)

In [ ]:
#датасет
df = pd.read_excel('/content/horeca_recruitment_analytics_28.01.2026-26.02.2026.xlsx')
df.head()

,Менеджер,id менеджера,Спецпризнак 1,Спецпризнак 2,Статус,"Создано вакансий, шт.","Потрачено публикаций, шт.","Публикация вакансии Стандарт, шт.","Публикация вакансии Стандарт плюс, шт.","Публикация вакансии Премиум, шт.",...,"Траты на контакты из базы резюме, руб.","Списано контактов из базы резюме, шт.","Стоимость контакта из базы резюме, руб.","Траты на публикации и контакты, руб.","Получено контактов (из откликов и базы резюме), шт.","Совокупная стоимость контакта, руб.","Доля откликов, шт.","Доля контактов, шт.","Доля откликов, руб.","Доля контактов, руб."
0,Иванова М.С.,1000,Панорама,Санкт-Петербург,Активен,5,12,9,2,1,...,0,0,0,2450,48,51.04,100.0,0.0,100.0,0.0
1,Иванова М.С.,1000,Брусника,Санкт-Петербург,Активен,3,8,6,1,1,...,0,0,0,2720,32,85.00,100.0,0.0,89.3,0.0
2,Иванова М.С.,1000,Усадьба,Великий Новгород,Активен,7,18,14,3,1,...,0,0,0,4180,89,46.97,100.0,0.0,100.0,0.0
3,Иванова М.С.,1000,Волна,Санкт-Петербург,Активен,4,11,9,2,0,...,1200,20,60,3530,72,49.03,72.2,27.8,66.0,34.0
4,Иванова М.С.,1000,Теремок,Великий Новгород,Активен,6,15,12,2,1,...,0,0,0,4440,78,56.92,100.0,0.0,73.0,0.0


Проделываем EDA

In [ ]:
#пропуски
print("ПРОПУСКИ")
nulls = df.isnull().sum()
print(nulls[nulls > 0])
if nulls.sum() == 0:
    print("Пропусков нет")
print()

ПРОПУСКИ
Индекс вежливости    20
dtype: int64



In [ ]:
#дубликаты
print("ДУБЛИКАТЫ")
dup_rows = df.duplicated().sum()
dup_key = df.duplicated(subset=['Менеджер', 'Спецпризнак 1', 'Спецпризнак 2']).sum()
print(f"Полных дубликатов строк: {dup_rows}")
print(f"Дубликатов по менеджер+ресторан+город: {dup_key}")
print()

ДУБЛИКАТЫ
Полных дубликатов строк: 0
Дубликатов по менеджер+ресторан+город: 0



In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 33 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Менеджер                                             299 non-null    object 
 1   id менеджера                                         299 non-null    int64  
 2   Спецпризнак 1                                        299 non-null    object 
 3   Спецпризнак 2                                        299 non-null    object 
 4   Статус                                               299 non-null    object 
 5   Создано вакансий, шт.                                299 non-null    int64  
 6   Потрачено публикаций, шт.                            299 non-null    int64  
 7   Публикация вакансии Стандарт, шт.                    299 non-null    int64  
 8   Публикация вакансии Стандарт плюс, шт.               299 non-null    i

In [ ]:
df.describe()

,id менеджера,"Создано вакансий, шт.","Потрачено публикаций, шт.","Публикация вакансии Стандарт, шт.","Публикация вакансии Стандарт плюс, шт.","Публикация вакансии Премиум, шт.","Публикация вакансии Оплата за отклики, шт.","Траты на публикации, руб.","Траты на размещение на Зарплата.ру, руб.",Количество трат на размещение на Зарплата.ру,...,"Траты на контакты из базы резюме, руб.","Списано контактов из базы резюме, шт.","Стоимость контакта из базы резюме, руб.","Траты на публикации и контакты, руб.","Получено контактов (из откликов и базы резюме), шт.","Совокупная стоимость контакта, руб.","Доля откликов, шт.","Доля контактов, шт.","Доля откликов, руб.","Доля контактов, руб."
count,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.0,299.000000,299.000000,299.000000,...,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000
mean,1014.451505,5.247492,14.026756,11.110368,2.140468,0.775920,0.0,2993.645485,278.929766,0.929766,...,194.648829,3.244147,12.642140,3467.224080,67.792642,54.688963,96.675920,2.979599,89.970903,3.655518
std,8.643585,1.977801,4.892737,3.869238,0.843650,0.417674,0.0,1074.184139,527.553736,1.758512,...,416.551189,6.942520,24.509463,1460.300369,33.623215,16.009360,6.898669,6.390263,12.737259,7.697223
min,1000.000000,2.000000,6.000000,5.000000,1.000000,0.000000,0.0,1250.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1250.000000,17.000000,38.480000,72.200000,0.000000,46.400000,0.000000
25%,1007.000000,4.000000,10.500000,8.500000,2.000000,1.000000,0.0,2245.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2330.000000,44.500000,46.070000,100.000000,0.000000,79.100000,0.000000
50%,1014.000000,5.000000,14.000000,11.000000,2.000000,1.000000,0.0,2970.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2970.000000,60.000000,50.230000,100.000000,0.000000,100.000000,0.000000
75%,1022.000000,7.000000,18.000000,14.000000,3.000000,1.000000,0.0,3980.000000,300.000000,1.000000,...,0.000000,0.000000,0.000000,4850.000000,89.000000,57.195000,100.000000,0.000000,100.000000,0.000000
max,1029.000000,9.000000,25.000000,20.000000,4.000000,1.000000,0.0,5360.000000,2100.000000,7.000000,...,2400.000000,40.000000,60.000000,7760.000000,178.000000,141.030000,100.000000,27.800000,100.000000,34.600000


In [ ]:
#категориальные колонки
cat_cols = ['Менеджер', 'Спецпризнак 1', 'Спецпризнак 2', 'Статус']
cat_cols = [c for c in cat_cols if c in df.columns]

print("КАТЕГОРИАЛЬНЫЕ ПЕРЕМЕННЫЕ")
for c in cat_cols:
    print(f"\n{c}:")
    print(df[c].value_counts(dropna=False).head(15))
print()

КАТЕГОРИАЛЬНЫЕ ПЕРЕМЕННЫЕ

Менеджер:
Менеджер
Иванова М.С.      10
Петрова А.В.      10
Сидорова Е.К.     10
Козлова О.Н.      10
Морозова Т.И.     10
Новикова И.П.     10
Волкова С.А.      10
Соколова Н.Д.     10
Лебедева М.О.     10
Кузнецова Е.В.    10
Попова А.С.       10
Семёнова Ю.Р.     10
Васильева Д.И.    10
Михайлова К.А.    10
Федорова Л.Н.     10
Name: count, dtype: int64

Спецпризнак 1:
Спецпризнак 1
Панорама    30
Брусника    30
Усадьба     30
Волна       30
Теремок     30
Заря        30
Гнездо      30
Ласточка    30
Садко       30
Берёзка     29
Name: count, dtype: int64

Спецпризнак 2:
Спецпризнак 2
Санкт-Петербург     179
Великий Новгород    120
Name: count, dtype: int64

Статус:
Статус
Активен         279
Заблокирован     20
Name: count, dtype: int64



________________________________________________________________________________
# Загрузка и описательная статистика для числовых данных - ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда эти данные и ответы модели

In [ ]:
#числовые колонки для описательной статистики
numeric_cols = [
    'Создано вакансий, шт.', 'Потрачено публикаций, шт.',
    'Траты на публикации, руб.', 'Индекс вежливости', 'Отклики, шт.',
    'Cтоимость отклика, руб.', 'Откликов на публикацию, шт.',
    'Получено контактов (из откликов и базы резюме), шт.',
    'Совокупная стоимость контакта, руб.'
]
#приводим колонки к числу (на случай запятых/точек)
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '.'), errors='coerce')

stats = df[numeric_cols].describe().round(0).astype(int) #окргулим, чтобы юзер не испугался
display(stats)

,"Создано вакансий, шт.","Потрачено публикаций, шт.","Траты на публикации, руб.",Индекс вежливости,"Отклики, шт.","Cтоимость отклика, руб.","Откликов на публикацию, шт.","Получено контактов (из откликов и базы резюме), шт.","Совокупная стоимость контакта, руб."
count,299,299,299,279,299,299,299,299,299
mean,5,14,2994,67,65,49,4,68,55
std,2,5,1074,11,29,8,1,34,16
min,2,6,1250,43,17,38,3,17,38
25%,4,10,2245,59,44,44,4,44,46
50%,5,14,2970,67,59,47,4,60,50
75%,7,18,3980,76,86,51,5,89,57
max,9,25,5360,91,138,79,6,178,141


In [ ]:
#текст описательной статистики для LLM
stats_text = stats.to_string()

response = client.chat.completions.create(
    model="gpt-4o-mini",  #или другая модель
    messages=[
        {"role": "user", "content": f"""Ты HR-аналитик. Объясни менеджеру, что означают эти цифры по подбору персонала. Говори по-русски, 2–3 абзаца.

ОПИСАТЕЛЬНАЯ СТАТИСТИКА:
{stats_text}

Объясни: всё ли ок, есть ли риски, на что смотреть."""}
    ]
)

display(response.choices[0].message.content)

'По представленным данным по подбору персонала можно сделать несколько выводов о текущей ситуации в процессе рекрутинга. Среднее количество созданных вакансий составляет 5, при этом максимальное значение достигает 9. Это говорит о том, что в большинстве случаев компания активно открывает новые позиции, однако стоит следить за тем, чтобы количество вакансий не превышало разумные пределы, иначе это может привести к перегрузке HR-отдела.\n\nПо количеству откликов на вакансии в среднем фиксируется 65 с стоимостью отклика около 2994 руб. Однако, учитывая максимальные значения, можно заметить, что некоторые вакансии получают до 138 откликов, а стоимость отклика для отдельных случаев может увеличиваться до 5360 руб. Это может говорить о том, что для определенных позиций требуется больше ресурсов на привлечение кандидатов, что может увеличивать общие затраты на подбор. Рекомендуется проанализировать наиболее дорогие вакансии и выявить, почему стоимость отклика такая высокая.\n\nТакже стоит обр

# **IQR.  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда эти данные и ответы модели**

In [ ]:
#выбросы (IQR)
print("Потенциальные выбросы (IQR)")
iqr_text_parts = []
for col in ['Cтоимость отклика, руб.', 'Траты на публикации, руб.']:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = df[(df[col] < lower) | (df[col] > upper)]
        line = f"{col}: {len(outliers)} выбросов (ниже {lower:.0f} или выше {upper:.0f} руб.)"
        print(line)
        iqr_text_parts.append(line)
        if len(outliers) > 0:
            sample = outliers[[col, 'Менеджер', 'Спецпризнак 1', 'Спецпризнак 2']].head(5).to_string()
            iqr_text_parts.append(f"Примеры:\n{sample}")

iqr_text = "\n".join(iqr_text_parts)
print()

Потенциальные выбросы (IQR)
Cтоимость отклика, руб.: 36 выбросов (ниже 33 или выше 62 руб.)
Траты на публикации, руб.: 0 выбросов (ниже -358 или выше 6582 руб.)



In [ ]:
#объяснение IQR от LLM для HR
response_iqr = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": f"""Ты HR-аналитик. Нам нашли статистические выбросы в данных по подбору. Объясни менеджеру, что это значит и что делать. Говори по-русски, без формул. 2–3 абзаца.

ВАЖНО: Все данные — только с HH.ru, других каналов нет. Выбросы — это разница между рекрутерами, спецпризнаками и городами.

РЕЗУЛЬТАТЫ АНАЛИЗА ВЫБРОСОВ (IQR):
{iqr_text}

Объясни: что означают выбросы в нашем случае, почему у одних рекрутеров/спецпризнаками/городов такие аномалии, на что обратить внимание."""}
    ]
)

display(response_iqr.choices[0].message.content)

'Выбросы в данных по подбору означают, что у нас есть значительные отклонения от средних показателей в стоимости отклика. В вашем случае это означает, что некоторые рекрутеры или спецификации подают заявки на отклик по ценам, которые значительно отличаются от большинства. Например, стоимость отклика может быть как крайне низкой (ниже 33 руб.) или значительно выше (выше 62 руб.). Это может свидетельствовать о различных причинах: возможно, некоторые рекрутеры используют менее эффективные методы поиска кандидатов, или же конкретные спецификации или регионы имеют разные уровни конкуренции на рынке труда, что влияет на стоимость отклика.\n\nВажно обратить внимание на тех рекрутеров, чьи показатели о стоимости отклика значительно выше или ниже среднего. Необходимо проанализировать, какие методы они используют для привлечения откликов, и как это соотносится с их результатами. Возможно, стоит провести обучение для тех, кто показывает низкие результаты, или, наоборот, изучить лучшие практики ус

# Срез: Рейтинг менеджеров по стоимости отклика.  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: у кого ниже стоимость привлечения кандидата, тот эффективнее по бюджету.

In [ ]:
#агрегат по менеджерам
manager_cost = df.groupby('Менеджер').agg(
    cost_mean=('Cтоимость отклика, руб.', 'mean'),
    responses=('Отклики, шт.', 'sum')
).reset_index()

#сортируем: самые экономные сверху
manager_cost = manager_cost.sort_values('cost_mean', ascending=True).round(0)

fig = px.bar(
    manager_cost,
    x='cost_mean',
    y='Менеджер',
    orientation='h',
    title='Эффективность HR-менеджеров: средняя стоимость отклика (руб.)',
    labels={'cost_mean': 'Стоимость отклика, руб.', 'Менеджер': ''},
    color='cost_mean',
    color_continuous_scale='RdYlGn_r'  # зелёный = дёшево, красный = дорого
)
fig.update_layout(showlegend=False, height=600)
fig.show()

In [ ]:
#текстовый свод для LLM (данные графика)
cost_summary = manager_cost.to_string()
top3 = manager_cost.head(3).to_string()
bot3 = manager_cost.tail(3).to_string()

prompt_cost = f"""Ты HR-аналитик. Объясни менеджеру или HRD эффективность рекрутеров по стоимости отклика. Говори по-русски, 2–3 абзаца.

ВАЖНО: Данные только с HH.ru.

РЕЙТИНГ МЕНЕДЖЕРОВ (средняя стоимость отклика, руб.):
{cost_summary}

Лучшие 3 (самая низкая стоимость):
{top3}

Худшие 3 (самая высокая стоимость):
{bot3}

Объясни: что видно из рейтинга, на кого обратить внимание, какие выводы."""

response_cost = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_cost}]
)

display(response_cost.choices[0].message.content)

'Исходя из предоставленных данных, мы можем проанализировать эффективность рекрутеров по критерию стоимости отклика. Наилучшие показатели демонстрируют менеджеры с наименьшими значениями среднего costs_per_response (например, Иванова М.С. – 47.0 руб.). Это подразумевает, что они более эффективно привлекают кандидатов, затрачивая меньше ресурсов на каждую заявку. Они получают хороший отклик при относительно низких затратах, что свидетельствует о высоком уровне компетенции и способности находить подходящих кандидатов.\n\nСравнивая эти данные с показателями менеджеров с высокой стоимостью отклика, такими как Макарова Т.В. (54.0 руб.), мы видим, что их затраты на привлечение кандидатов значительно выше, что может указывать на необходимость повышения эффективности их работы. Эти менеджеры, возможно, тратят больше ресурсов на поиск, но получают меньше откликов, что может быть сигналом для тренингов или пересмотра стратегий подбора.\n\nВ целом, важно обратить внимание на опыт и методы работы 

# Срез: По статусам.  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: активные vs заблокированные менеджеры, их объём и метрики.

In [ ]:
status_metrics = df.groupby('Статус').agg(
    managers=('Менеджер', 'nunique'),
    responses=('Отклики, шт.', 'sum'),
    cost_mean=('Cтоимость отклика, руб.', 'mean'),
    spend_total=('Траты на публикации, руб.', 'sum')
).reset_index()

fig = px.bar(
    status_metrics,
    x='Статус',
    y='responses',
    color='Статус',
    title='Отклики по статусу менеджеров',
    labels={'responses': 'Всего откликов'}
)
fig.update_layout(showlegend=False, height=400)
fig.show()

In [ ]:
status_text = status_metrics.round(2).to_string(index=False)
prompt_status = f"""Ты HR-аналитик. Объясни менеджеру, что видно по статусам рекрутеров. 2–3 абзаца.

ДАННЫЕ:
{status_text}

Объясни: чем отличаются активные и заблокированные, что это даёт для управления подбором."""

response_status = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt_status}])
display(response_status.choices[0].message.content)

'Статусы рекрутеров дают ключевую информацию о текущем состоянии процесса подбора персонала. В нашем случае из анализа видно, что активные рекрутеры (28 человек) значительно превышают количество заблокированных (2 человека) по всем показателям. Они провели 17,960 взаимодействий с менеджерами, что свидетельствует о высокой активности и вовлеченности в процесс поиска и подбора кандидатов. Средняя стоимость на одно взаимодействие у активных и заблокированных рекрутеров отличается незначительно – 49.16 и 49.20 соответственно, однако общий объем затрат на активные рекрутера составляет 834,420, что указывает на их значительный вклад в общую эффективность подбора.\n\nРазница между активными и заблокированными рекрутерами, помимо количества взаимодействий, также указывает на потенциальные проблемы в работе заблокированных сотрудников. Заблокированные рекрутеры (всего 2) имеют значительно меньший объем взаимодействий – 1,340, что может говорить о недостаточной квалификации, низкой мотивации или

# Срез: По статусам.  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: сколько всего публикаций на менеджера по типам (Стандарт + Стандарт+ + Премиум + Оплата за отклики).

In [ ]:
pub_cols = ['Публикация вакансии Стандарт, шт.', 'Публикация вакансии Стандарт плюс, шт.',
            'Публикация вакансии Премиум, шт.', 'Публикация вакансии Оплата за отклики, шт.']
pub_cols = [c for c in pub_cols if c in df.columns]

manager_pub = df.groupby('Менеджер')[pub_cols].sum().reset_index()
for c in pub_cols:
    manager_pub[c] = pd.to_numeric(manager_pub[c], errors='coerce')

manager_pub['Всего'] = manager_pub[pub_cols].sum(axis=1)

pct_cols = {}
for c in pub_cols:
    short = c.replace('Публикация вакансии ', '').replace(', шт.', '')
    pct_cols[short] = (manager_pub[c] / manager_pub['Всего'] * 100).round(1)

manager_pub_pct = manager_pub[['Менеджер']].copy()
for k, v in pct_cols.items():
    manager_pub_pct[k] = v

manager_pub_pct_long = manager_pub_pct.melt(
    id_vars='Менеджер',
    value_vars=list(pct_cols.keys()),
    var_name='Тип',
    value_name='Доля, %'
)

fig = px.bar(
    manager_pub_pct_long,
    x='Менеджер',
    y='Доля, %',
    color='Тип',
    barmode='stack',
    title='Доли типов публикаций по менеджерам (%)'
)
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

In [ ]:
pub_text = manager_pub_pct.round(2).to_string(index=False)

prompt_pub = f"""Ты HR-аналитик. Объясни менеджеру структуру публикаций на HH.ru. Говори по-русски, 2–3 абзаца.

Данные — доли типов публикаций по рекрутерам (Стандарт, Стандарт плюс, Премиум, Оплата за отклики). Чем дороже тип, тем выше стоимость отклика.

ДАННЫЕ:
{pub_text}

Объясни: как у кого распределены типы публикаций, где перекос в сторону дорогих или дешёвых, какие выводы и рекомендации."""

response_pub = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_pub}]
)
display(response_pub.choices[0].message.content)

'На основе предоставленных данных можно наблюдать распределение типов публикаций среди рекрутеров, представленных в таблице. Все рекрутеры в основном используют тип публикации "Стандарт" (в среднем около 79%), за ним следует "Стандарт плюс" (около 15%) и тип "Премиум" (примерно 5.5%). Видно, что ни один из рекрутеров не использует опцию "Оплата за отклики". Это говорит о том, что большинство менеджеров предпочитают более экономичные варианты, при этом только небольшая доля публикаций делается в дорогих категориях.\n\nВажно отметить, что среди всех рекрутеров наблюдается перекос в сторону более дешевых типов публикаций. Например, Александрова Т.С. имеет 78.8% публикаций "Стандарт", и только 5.8% — "Премиум", что указывает на отсутствие стратегии привлечения кандидатов с использованием более дорогих и, вероятно, более целевых форматов. Это может объяснять низкое разнообразие откликов.\n\nРекомендуется провести анализ эффективности текущих публикаций. Возможно, для некоторых позиций стоит

# Срез: Приглашения из откликов и из базы резюме.  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: сравнение приглашений из откликов и из базы резюме, из поиска по менеджерам

In [ ]:
#подбираем колонки, связанные с приглашениями
inv_cols = [c for c in df.columns if 'Приглаш' in c or 'приглаш' in c.lower()]
#оставляем только те, что содержат единицы (шт.)
inv_cols = [c for c in inv_cols if 'шт' in c or 'шт.' in c]
#если не нашли —- пробуем по частичному совпадению
if not inv_cols:
    inv_cols = [c for c in df.columns if 'приглаш' in c.lower()]

if inv_cols:
    #приводим колонки к числу
    for c in inv_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    #считаем сумму приглашений по каждому источнику для каждого менеджера
    manager_inv = df.groupby('Менеджер')[inv_cols].sum().reset_index()
    #добавляем общее число приглашений
    manager_inv['Всего приглашений'] = manager_inv[inv_cols].sum(axis=1)
    #сортируем по убыванию
    manager_inv = manager_inv.sort_values('Всего приглашений', ascending=False)


    inv_short = [c.replace('Приглашений из ', '').replace('Приглашений ', '').replace(', шт.', '').strip() for c in inv_cols]
    # Преобразуем в «длинный» формат: один столбец — источник, другой — количество
    manager_inv_long = manager_inv.melt(
        id_vars='Менеджер',      #столбец, по которому группируем
        value_vars=inv_cols,    #колонки, которые «разворачиваем»
        var_name='Источник',    #название нового столбца с источником
        value_name='Кол-во'     #название столбца с числами
    )
    #замена
    manager_inv_long['Источник'] = manager_inv_long['Источник'].replace(dict(zip(inv_cols, inv_short)))

    #stacked bar chart
    fig = px.bar(
        manager_inv_long,
        x='Менеджер',           # ось X
        y='Кол-во',             # ось Y (значения)
        color='Источник',       # цвет = тип источника
        barmode='stack',        # столбцы накладываются друг на друга
        title='Приглашения по менеджерам (все источники)',
        labels={'Кол-во': 'Приглашений'}
    )
    #наклон подписей по оси X
    fig.update_layout(height=500, xaxis_tickangle=-45)
    fig.show()
else:
    print('Колонки с приглашениями не найдены')

In [ ]:
if inv_cols:
    #свод данных в текстовый вид для промпта
    inv_text = manager_inv.round(2).to_string(index=False)
    sources_list = ', '.join(inv_short)

    prompt_inv = f"""Ты HR-аналитик. Объясни менеджеру, как рекрутеры используют каналы приглашений на собеседования. Говори по-русски, 2–3 абзаца.

В выгрузке есть приглашения по источникам: {sources_list}. Данные с HH.ru.

ДАННЫЕ:
{inv_text}

Объясни: как распределены каналы, у кого перекос, какие выводы и рекомендации."""

    response_inv = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_inv}]
    )
    display(response_inv.choices[0].message.content)

'В представленных данных мы видим количество приглашений на собеседования, сделанных рекрутерами через два основных канала: отклики соискателей и базы резюме. По данным видно, что большинство приглашений (в среднем около 23-25%) исходят из базы резюме, в то время как гораздо больший процент (примерно 75-77%) формируется из откликов. Например, менеджер Петрова А.В. сделала 239 приглашений из откликов и только 41 из базы резюме, что свидетельствует о явном перекосе в использовании этих каналов.\n\nТакой перекос в разработке стратегий поиска может говорить о том, что рекрутеры больше полагаются на активные отклики соискателей, нежели на проактивный поиск кандидатов через базы. Это может указывать на упущенные возможности, так как базы резюме могут содержать более качественных и подходящих кандидатов, которые могли не откликнуться на текущие вакансии. Рекомендуем фокусироваться на балансировке ресурсов, развивать стратегии проактивного поиска и активно использовать базы резюме для поиска у

# Срез: Публикации (сумма типов).  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: сколько всего публикаций на менеджера по типам (Стандарт + Стандарт+ + Премиум + Оплата за отклики).

In [ ]:
#фильтруем только активных и где ИВ есть
df_active = df[(df['Статус'] == 'Активен') & (df['Индекс вежливости'].notna())]
df_active['Индекс вежливости'] = pd.to_numeric(df_active['Индекс вежливости'], errors='coerce')

manager_iv = df_active.groupby('Менеджер').agg(
    iv_mean=('Индекс вежливости', 'mean'),
    responses=('Отклики, шт.', 'sum')
).reset_index()
manager_iv = manager_iv.sort_values('iv_mean', ascending=False)  #сверху вниз

fig = px.bar(
    manager_iv,
    x='Менеджер',
    y='iv_mean',
    title='Индекс вежливости: как полно обрабатываются отклики (%)',
    labels={'iv_mean': 'Индекс вежливости, %', 'Менеджер': ''},
    color='iv_mean',
    color_continuous_scale='Greens'
)
fig.update_layout(showlegend=False, height=500, xaxis_tickangle=-45)
fig.show()

/tmp/ipython-input-588/458024223.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
iv_summary = manager_iv.to_string()
iv_top3 = manager_iv.head(3).to_string()
iv_bot3 = manager_iv.tail(3).to_string()

prompt_iv = f"""Ты HR-аналитик. Объясни менеджеру значение индекса вежливости (ИВ) и работу рекрутеров. Говори по-русски, 2–3 абзаца.

ВАЖНО — методика HH.ru: ИВ = скорость ответа + доля просмотренных откликов. Показывает, насколько продуктивно тратятся деньги на публикации: сколько откликов обработано, а сколько — «вхолостую». Норма — от 70%, хорошо — от 90%. Ниже 70% — зона внимания.

РЕЙТИНГ МЕНЕДЖЕРОВ (средний ИВ, %):
{iv_summary}

Лучшие 3:
{iv_top3}

Худшие 3 (ниже нормы или близко к ней):
{iv_bot3}

Объясни: что значит ИВ в контексте HH.ru, у кого проблемы, какие действия предложить."""

response_iv = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_iv}]
)
display(response_iv.choices[0].message.content)

'Индекс вежливости (ИВ) на платформе HH.ru — это важный метрик, который отражает, насколько эффективно и продуктивно используются средства на публикации вакансий. Он рассчитывается как сумма скорости ответа и доли просмотренных откликов, что позволяет оценить, сколько откликов было действительно обработано, а сколько потеряно без внимания. Для успешной работы рекрутеров важно поддерживать ИВ на уровне не ниже 70%, а идеальный показатель превышает 90%. Это значит, что чем выше ИВ, тем более эффективно ведется поиск кандидатов и работа с откликами.\n\nВ вашей команде наблюдаются некоторые проблемы, особенно у менеджеров с низким ИВ. Например, Макарова Т.В., Киселёва Н.М. и Павлова И.С. имеют показатели ниже 70%, что сигнализирует о необходимости особого внимания к процессу обработки откликов. Возможно, причина в недостаточной скорости реагирования на кандидатов или в том, что многие отклики не были просмотрены. Рекомендуется провести анализ их работы, выявить узкие места в их процессе и 

# Срез: Просмотры резюме. ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели
Смысл: просмотры из откликов vs просмотры из поиска.

In [ ]:
view_cols = [c for c in df.columns if 'Просмотр' in c and 'резюме' in c.lower()]
view_cols = [c for c in view_cols if c in df.columns]

if view_cols:
    for c in view_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    manager_views = df.groupby('Менеджер')[view_cols].sum().reset_index()
    manager_views['Всего просмотров'] = manager_views[view_cols].sum(axis=1)

    view_short = [
        c.replace('Просмотры резюме ', '').replace('из ', '').replace(', шт.', '').strip()
        for c in view_cols
    ]

    for c, short in zip(view_cols, view_short):
        manager_views[short] = (manager_views[c] / manager_views['Всего просмотров'] * 100).round(1)

    manager_views_long = manager_views.melt(
        id_vars='Менеджер',
        value_vars=view_short,
        var_name='Источник',
        value_name='Доля, %'
    )

    fig = px.area(
        manager_views_long,
        x='Менеджер',
        y='Доля, %',
        color='Источник',
        line_group='Источник',
        title='Доли просмотров резюме по источникам (%) — волновой график'
    )
    fig.update_layout(
        height=500,
        xaxis_tickangle=-45,
        legend_title='Источник',
        yaxis=dict(range=[0, 100], ticksuffix='%')
    )
    fig.show()

In [ ]:
if view_cols:
    views_text = manager_views[['Менеджер'] + view_short].round(2).to_string(index=False)
    sources_list = ', '.join(view_short)

    prompt_views = f"""Ты HR-аналитик. Объясни менеджеру просмотры резюме и как читать волновой график. 2–3 абзаца.

КАК ЧИТАТЬ ГРАФИК: по оси X — рекрутеры, по оси Y — доли в процентах. Цветные слои показывают, из какого источника пришли просмотры: {sources_list}. Сумма по каждому менеджеру = 100%. Чем выше слой — тем больше доля этого канала. Узкое место или перекос между слоями — зона внимания.

ДАННЫЕ (доли %):
{views_text}

Объясни: как интерпретировать график, у кого какой баланс источников, какие выводы и рекомендации."""

    response_views = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_views}]
    )
    display(response_views.choices[0].message.content)

'График, показывающий просмотры резюме, позволяет визуализировать и анализировать, откуда приезжают кандидаты для разных менеджеров. По оси X мы видим имена рекрутеров, а по оси Y – доли процентов, которые указывают на распределение источников: поиск, отклик и просмотры резюме. Каждый слоеведет к пониманию того, какой канал привлечения привносит больше всего кандидатур для рекрутера. Чем выше слой, тем больше доля этого источника. Важно обратить внимание на узкие места и перекосы между слоями – это может указывать на неравномерное использование источников или недостаточного привлечения кандидатов через определенные каналы.\n\nПо представленным данным, все менеджеры имеют одинаковую долю просмотров резюме на уровне 50%. Баланс между источниками ("поиск" и "отклик") также отличается лишь незначительно, однако наблюдаются некоторые отклонения. Например, у Андреевой В.М. и Ивановой М.С. более высокий процент откликов (26.2% и 27.0% соответственно), что может означать более высокую активнос

# Срез: Продуктивность публикаций (откликов на публикацию).  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: сколько откликов приносит каждая трата публикации. Чем выше — тем эффективнее использование публикаций.

In [ ]:
manager_prod = df.groupby('Менеджер').agg(
    prod_mean=('Откликов на публикацию, шт.', 'mean'),
    pub_spent=('Потрачено публикаций, шт.', 'sum'),
    responses=('Отклики, шт.', 'sum')
).reset_index()
manager_prod = manager_prod.sort_values('prod_mean', ascending=False).round(2)

fig = px.bar(
    manager_prod,
    x='Менеджер',
    y='prod_mean',
    title='Продуктивность публикаций: откликов на 1 трату',
    labels={'prod_mean': 'Откликов на публикацию', 'Менеджер': ''},
    color='prod_mean',
    color_continuous_scale='Blues'
)
fig.update_layout(showlegend=False, height=500, xaxis_tickangle=-45)
fig.show()

In [ ]:
prod_summary = manager_prod.to_string()
prod_top3 = manager_prod.head(3).to_string()
prod_bot3 = manager_prod.tail(3).to_string()

prompt_prod = f"""Ты HR-аналитик. Объясни менеджеру продуктивность публикаций. Говори по-русски, 2–3 абзаца.

ВАЖНО (методика HH.ru): продуктивность = отклики / потраченные публикации. Сколько кандидатов даёт каждая трата. Чем выше — тем лучше использование бюджета на HH.ru.

РЕЙТИНГ МЕНЕДЖЕРОВ:
{prod_summary}

Лучшие 3:
{prod_top3}

Худшие 3:
{prod_bot3}

Объясни: что видно по продуктивности, у кого проблемы, какие рекомендации."""

response_prod = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_prod}]
)
display(response_prod.choices[0].message.content)

'Анализ продуктивности публикаций среди менеджеров показывает заметное расхождение в результатах. Наилучшие показатели продемонстрировали Иванова М.С. (продуктивность 4.61), Новикова И.П. (4.55) и Сидорова Е.К. (4.53). Эти менеджеры обеспечивают высокое количество откликов при относительно низких затратах на публикации, что свидетельствует о грамотном использовании бюджета и эффективной стратегии подбора кандидатов. Они, вероятно, используют лучшие практики в формулировке вакансий и привлекают более целевую аудиторию.\n\nСреди менеджеров с худшей продуктивностью выделяются Дмитриева Е.А. (4.27), Павлова И.С. (4.19) и Макарова Т.В. (4.12). Низкая эффективность их публикаций может указывать на недостаточно привлекательные или нечетко сформулированные вакансии, а также на возможные проблемы в таргетировании аудитории. Рекомендуется проанализировать описания их вакансий и стратегии взаимодействия с кандидатами для выявления слабых мест. Дополнительно, стоит рассмотреть возможность обучения

# Срез: Конверсия отклик /  приглашение.  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: доля откликов, превращённых в приглашения.

In [ ]:
#поиск колонок
col_resp = [c for c in df.columns if 'Отклики' in c and ('шт' in c or 'шт.' in c)]
col_inv = [c for c in df.columns if 'Приглаш' in c]
col_inv = [c for c in col_inv if c in df.columns]

if col_resp and col_inv:
    for c in col_resp + col_inv:
        df[c] = pd.to_numeric(df[c], errors='coerce') #чтобы не обосраться

    manager_conv = df.groupby('Менеджер').agg(
        responses=(col_resp[0], 'sum'),
        invites=(col_inv[0], 'sum')
    ).reset_index()
    manager_conv.columns = ['Менеджер', 'Отклики', 'Приглашения']
    manager_conv['Конверсия, %'] = (manager_conv['Приглашения'] / manager_conv['Отклики'] * 100).round(0)
    manager_conv = manager_conv.sort_values('Конверсия, %', ascending=False)

    fig = px.bar(manager_conv, x='Менеджер', y='Конверсия, %', title='Конверсия отклик / приглашение (%)',
                 color='Конверсия, %', color_continuous_scale='Viridis')
    fig.update_layout(showlegend=False, height=500, xaxis_tickangle=-45)
    fig.show()

In [ ]:
if col_resp and col_inv:
    conv_text = manager_conv.round(2).to_string(index=False)
    prompt_conv = f"""Ты HR-аналитик. Объясни менеджеру конверсию откликов в приглашения. 2–3 абзаца.

Конверсия = сколько % откликов превращаются в приглашения. Показывает качество скрининга.

ДАННЫЕ:
{conv_text}

Объясни: у кого лучше/хуже скрининг, где зона внимания."""

    response_conv = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt_conv}])
    display(response_conv.choices[0].message.content)

'Конверсия откликов в приглашения является важным показателем эффективности процесса подбора кадров. В наших данных видно, что все перечисленные менеджеры имеют конверсию, равную 35% или 34%, что указывает на стандартный уровень скрининга. Это означает, что примерно треть откликов превращается в приглашения на собеседование для большинства менеджеров, что помогает оценить качество оценки кандидатов и точность соответствия их требованиям.\n\nОднако общее значение конверсии может скрывать индивидуальные различия. Например, среди менеджеров с конверсией 35% можно выделить Григорьеву О.И., которая отправила 624 отклика и получила 218 приглашений, тогда как, к примеру, у Романовой Е.Л. откликов было 670, но приглашений — 232. Важно заметить, что, несмотря на схожую конверсию, количество откликов и приглашений у разных менеджеров колеблется, и это может указывать на качество кандидатов, на которых они ориентируются. Для менеджеров с более низкими показателями (34% и 33%) - Лебедева М.О., Коз

# Срез: Сводный скор.  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: один комбинированный показатель по нескольким метрикам.



$$
\text{Скор} = \frac{\tilde{C}_{cost} + \tilde{C}_{IV} + \tilde{C}_{prod}}{3}
$$

Минимум–максимум нормализация в 0–100. Стоимость — инвертирована (меньше = лучше).

In [ ]:
manager_score = df.groupby('Менеджер').agg(
    cost=('Cтоимость отклика, руб.', 'mean'),
    iv=('Индекс вежливости', 'mean'),
    prod=('Откликов на публикацию, шт.', 'mean')
).reset_index()

for c in ['cost', 'iv', 'prod']:
    manager_score[c] = pd.to_numeric(manager_score[c], errors='coerce')

#нормализация 0–100: cost — меньше лучше, iv и prod — больше лучше
manager_score['cost_norm'] = 100 - (manager_score['cost'] - manager_score['cost'].min()) / (manager_score['cost'].max() - manager_score['cost'].min() + 1e-6) * 100
manager_score['iv_norm'] = (manager_score['iv'] - manager_score['iv'].min()) / (manager_score['iv'].max() - manager_score['iv'].min() + 1e-6) * 100
manager_score['prod_norm'] = (manager_score['prod'] - manager_score['prod'].min()) / (manager_score['prod'].max() - manager_score['prod'].min() + 1e-6) * 100

manager_score['Скор'] = (manager_score['cost_norm'] + manager_score['iv_norm'] + manager_score['prod_norm']) / 3
manager_score = manager_score.sort_values('Скор', ascending=False)

fig = px.bar(manager_score, x='Менеджер', y='Скор', title='Сводный скор (стоимость + ИВ + продуктивность)',
             color='Скор', color_continuous_scale='RdYlGn')
fig.update_layout(showlegend=False, height=500, xaxis_tickangle=-45)
fig.show()

In [ ]:
print("""
СВОДНЫЙ СКОР
=============
Скор = (cost_norm + iv_norm + prod_norm) / 3

Компоненты (0–100):
- cost_norm = 100 × (1 − (cost − min) / (max − min)) — чем меньше стоимость, тем лучше
- iv_norm = 100 × (iv − min) / (max − min) — чем выше ИВ, тем лучше
- prod_norm = 100 × (prod − min) / (max − min) — чем выше продуктивность, тем лучше
""")


СВОДНЫЙ СКОР
Скор = (cost_norm + iv_norm + prod_norm) / 3

Компоненты (0–100):
- cost_norm = 100 × (1 − (cost − min) / (max − min)) — чем меньше стоимость, тем лучше
- iv_norm = 100 × (iv − min) / (max − min) — чем выше ИВ, тем лучше
- prod_norm = 100 × (prod − min) / (max − min) — чем выше продуктивность, тем лучше



In [ ]:
score_text = manager_score[['Менеджер', 'cost', 'iv', 'prod', 'Скор']].round(2).to_string(index=False)
prompt_score = f"""Ты HR-аналитик. Объясни менеджеру сводный скор рекрутеров. 2–3 абзаца.

Скор объединяет стоимость отклика (меньше — лучше), индекс вежливости и продуктивность (больше — лучше). 0–100.

ДАННЫЕ:
{score_text}

Объясни: кто лидирует, кто отстаёт, основные выводы."""

response_score = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt_score}])
display(response_score.choices[0].message.content)

'В сводном скоре рекрутеров в нашей команде мы видим значительные различия в эффективности работы сотрудников. Лидером по этому показателю стала Иванова М.С. с общим скором 98.45, что обусловлено относительно низкой стоимостью отклика (47.42), высоким индексом вежливости (72.05) и хорошей продуктивностью (4.61). Новикова И.П. и Петрова А.В. также демонстрируют высокие результаты с показателями скор 87.61 и 82.33 соответственно, однако их стоимость отклика чуть выше, что можно рассматривать как область для улучшения.\n\nСреди рекрутеров, которые отстают, выделяются такие сотрудники, как Павлова И.С. и Макарова Т.В., которые имеют крайне низкий скор – 13.11 и 1.96. Их высокая стоимость отклика (52.24 и 53.90 соответственно) вызывает серьезные сомнения в их эффективности, несмотря на приемлемые показатели вежливости и продуктивности. Важно, чтобы мы сосредоточились на выявлении причин низкой результативности этих рекрутеров и разработали план для повышения их производительности. В целом, 

# Срез: Выше/ниже среднего
Смысл: сравнение рекрутеров со средним по компании.

In [ ]:
#сводка
manager_vs = df.groupby('Менеджер').agg(
    cost=('Cтоимость отклика, руб.', 'mean'),
    iv=('Индекс вежливости', 'mean'),
    prod=('Откликов на публикацию, шт.', 'mean')
).reset_index()
for c in ['cost', 'iv', 'prod']:
    manager_vs[c] = pd.to_numeric(manager_vs[c], errors='coerce')

cost_avg = manager_vs['cost'].mean()
iv_avg = manager_vs['iv'].mean()
prod_avg = manager_vs['prod'].mean()

manager_vs['cost_color'] = ['#2ecc71' if x <= cost_avg else '#e74c3c' for x in manager_vs['cost']]
manager_vs['iv_color'] = ['#2ecc71' if x >= iv_avg else '#e74c3c' for x in manager_vs['iv']]
manager_vs['prod_color'] = ['#2ecc71' if x >= prod_avg else '#e74c3c' for x in manager_vs['prod']]

#cортируем по метрикам
m_cost = manager_vs.sort_values('cost', ascending=True).round(0)
m_iv = manager_vs.sort_values('iv', ascending=False).round(0)
m_prod = manager_vs.sort_values('prod', ascending=False).round(0)

#один типа-figure с тремя столбцами
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    f'1. Стоимость отклика (ср. {cost_avg:.0f} руб.)',
    f'2. Индекс вежливости (ср. {iv_avg:.1f}%)',
    f'3. Откликов на публикацию (ср. {prod_avg:.2f})'
), horizontal_spacing=0.08)

fig.add_trace(go.Bar(x=m_cost['Менеджер'], y=m_cost['cost'], marker_color=m_cost['cost_color'], name='стоимость'), row=1, col=1)
fig.add_trace(go.Bar(x=m_iv['Менеджер'], y=m_iv['iv'], marker_color=m_iv['iv_color'], name='ИВ'), row=1, col=2)
fig.add_trace(go.Bar(x=m_prod['Менеджер'], y=m_prod['prod'], marker_color=m_prod['prod_color'], name='продуктивность'), row=1, col=3)

fig.add_hline(y=cost_avg, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=iv_avg, line_dash="dash", line_color="gray", row=1, col=2)
fig.add_hline(y=prod_avg, line_dash="dash", line_color="gray", row=1, col=3)

fig.update_xaxes(tickangle=-45)
fig.update_layout(height=500, showlegend=False, title_text='Сравнение с средним по компании')
fig.show()

print("Зелёный — лучше среднего | Красный — хуже | Пунктир — среднее")

Зелёный — лучше среднего | Красный — хуже | Пунктир — среднее


In [ ]:
vs_text = manager_vs[['Менеджер', 'cost', 'iv', 'prod']].round(2).to_string(index=False)

prompt_vs = f"""Ты HR-аналитик. Объясни менеджеру, как читать график и что из него следует. 2–3 абзаца.

КАК ЧИТАТЬ: три панели рядом — стоимость отклика, индекс вежливости, продуктивность. По оси X — рекрутеры, по оси Y — значения метрики. Пунктирная линия — среднее по компании. Зелёный столбец — значение лучше среднего, красный — хуже. По стоимости «лучше» = ниже, по ИВ и продуктивности — выше.

ДАННЫЕ:
{vs_text}

Средние: стоимость {cost_avg:.0f} руб., ИВ {iv_avg:.1f}%, продуктивность {prod_avg:.2f}.

Объясни: как интерпретировать график, кто в плюсе/минусе по каждой метрике, какие выводы и рекомендации."""

response_vs = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_vs}]
)
display(response_vs.choices[0].message.content)

'На графике представлена информация о трех ключевых метриках работы рекрутеров: стоимости отклика, индексе вежливости (ИВ) и продуктивности. По оси X расположены имена рекрутеров, а по оси Y — значения метрик. Пунктирная линия обозначает средние показатели для компании. Зеленые столбцы указывают на результаты выше среднего, в то время как красные столбцы — на показатели ниже среднего. \n\nЧто касается интерпретации данных: в метрике «стоимость отклика» рекрутеры с зелеными столбцами (например, Иванова М.С. и Волкова С.А.) имеют более низкие значения, что является положительным показателем для компании. В то же время, Андреева В.М. и Макарова Т.В. находятся в красной зоне, что говорит о том, что их затраты на привлечение откликов превышают средние по компании. В индексе вежливости внимание стоит уделить Ивановой М.С. и Волковой С.А., демонстрирующим высокие показатели, в то время как рекрутеры с пропущенными данными (например, Андреева В.М. и Сидорова Е.К.) требуют дополнительного анали

# Срез: Траты на 1 контакт у менеджера на одно приглашение и один отклик . ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели
Cмысл: соотношение трат и результатов по воронке

In [ ]:
if col_spend and col_resp and inv_cols:
    manager_split['Руб. на отклик'] = (manager_split['Траты'] / manager_split['Откликов']).round(0)
    manager_split['Руб. на приглашение'] = (manager_split['Траты'] / manager_split['Приглашений']).round(0)

    m_resp = manager_split.sort_values('Руб. на отклик')
    m_inv = manager_split.sort_values('Руб. на приглашение')

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        'Стоимость 1 отклика (руб.) — ранняя стадия',
        'Стоимость 1 приглашения (руб.) — поздняя стадия'
    ), horizontal_spacing=0.1)

    fig.add_trace(go.Bar(x=m_resp['Менеджер'], y=m_resp['Руб. на отклик'],
                         text=m_resp['Руб. на отклик'], textposition='outside',
                         marker_color=m_resp['Руб. на отклик'], showlegend=False),
                  row=1, col=1)
    fig.add_trace(go.Bar(x=m_inv['Менеджер'], y=m_inv['Руб. на приглашение'],
                         text=m_inv['Руб. на приглашение'], textposition='outside',
                         marker_color=m_inv['Руб. на приглашение'], showlegend=False),
                  row=1, col=2)

    fig.update_xaxes(tickangle=-45)
    fig.update_layout(height=500, title_text='Стоимость по стадиям воронки')
    fig.show()

In [ ]:
if col_spend and col_resp and inv_cols:
    split_text = manager_split[['Менеджер', 'Траты', 'Откликов', 'Приглашений', 'Руб. на отклик', 'Руб. на приглашение']].round(2).to_string(index=False)

    prompt_split = f"""Ты HR-аналитик. Объясни менеджеру стоимость по стадиям воронки. 2–3 абзаца.

КАК ЧИТАТЬ ГРАФИК: слева — стоимость 1 отклика (ранняя стадия), справа — стоимость 1 приглашения (поздняя стадия).  Эти метрики не ROI, а удельная стоимость привлечения на каждом этапе.

ДАННЫЕ:
{split_text}

Объясни: как читать график, у кого дороже/дешевле на каждой стадии, какие выводы."""

    response_split = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_split}]
    )
    display(response_split.choices[0].message.content)

'Для анализа стоимости по стадиям воронки важно правильно интерпретировать данные. График имеет две ключевые метрики: стоимость одного отклика и стоимость одного приглашения. Стоимость на ранней стадии (отклики) отражает, сколько компания тратит на привлечение потенциальных кандидатов, тогда как стоимость на поздней стадии (приглашения) показывает, сколько стоит дальнейшая отборка кандидатов для собеседований. Эти показатели помогают нам оценить эффективность различных менеджеров по привлечению и отбору кандидатур.\n\nАнализируя данные, можно выделить несколько интересных моментов. Например, Александрова Т.С. и Васильева Д.И. имеют одинаковую стоимость на отклик — 52 рубля, что указывает на их эффективность в привлечении кандидатов. Однако стоимость приглашений у них различается: у Александровой — 150 рублей, а у Васильевой — 150 рублей, что показывает равную эффективность на последующей стадии. В то же время, Петрова А.В. имеет самые высокие затраты на отклик — 60 рублей, что может ук

# Срез: Эффективность по спецпризнак 2.  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Cмысл: сравнение представительств компании по ключевым метрикам.

In [ ]:
city_metrics = df.groupby('Спецпризнак 2').agg(
    cost_mean=('Cтоимость отклика, руб.', 'mean'),
    iv_mean=('Индекс вежливости', 'mean'),
    prod_mean=('Откликов на публикацию, шт.', 'mean'),
    responses=('Отклики, шт.', 'sum'),
    spend_total=('Траты на публикации, руб.', 'sum')
).round(0).reset_index()

#сводная таблица для графика
city_long = city_metrics.melt(
    id_vars='Спецпризнак 2',
    value_vars=['cost_mean', 'iv_mean', 'prod_mean'],
    var_name='Метрика',
    value_name='Значение'
)
city_long['Метрика'] = city_long['Метрика'].map({
    'cost_mean': 'Стоимость отклика, руб.',
    'iv_mean': 'Индекс вежливости, %',
    'prod_mean': 'Откликов на публикацию'
})

fig = px.bar(
    city_long,
    x='Спецпризнак 2',
    y='Значение',
    color='Метрика',
    barmode='group',
    title='Сравнение предстваительств компании',
    labels={'Спецпризнак 2': 'Город', 'Значение': ''}
)
fig.update_layout(height=500, xaxis_tickangle=0)
fig.show()

In [ ]:
city_summary = city_metrics.to_string()

prompt_city = f"""Ты HR-аналитик. Объясни менеджеру разницу между городами. Говори по-русски, 2–3 абзаца.

ВАЖНО: Все данные с HH.ru.

СВОДКА ПО ГОРОДАМ:
{city_summary}

Объясни: где подбор идёт легче, где дороже, в каком городе хуже индекс вежливости и о чём это говорит."""

response_city = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_city}]
)
display(response_city.choices[0].message.content)

'Сравнивая данные по городам, мы можем сделать несколько ключевых наблюдений. В Санкт-Петербурге стоимость подбора (cost_mean) составляет 48.0, что ниже, чем в Великом Новгороде, где она достигает 51.0. Это говорит о том, что подбор сотрудников в Санкт-Петербурге, в целом, обходится дешевле. Однако стоит обратить внимание на количество откликов (responses) – в Санкт-Петербурге их 12045, что значительно больше, чем в Великом Новгороде с 7255 откликами. Это может свидетельствовать о том, что в Санкт-Петербурге рынок труда более активный и привлекает больше кандидатов.\n\nЧто касается индекса вежливости (iv_mean), в Великом Новгороде он равен 72.0, что выше, чем в Санкт-Петербурге, где этот показатель составляет 64.0. Более высокий индекс вежливости в Великом Новгороде может указывать на то, что кандидаты более отзывчивы и лучше реагируют на вакансии, чем в Санкт-Петербурге. Это может быть связано с культурными особенностями региона или различиями в уровне конкуренции между работодателями

# Срез: Эффективность по спецпризнак 1 .  ПОКАЗЫВАЕМ В ВЫВОДЕ Дашборда график и ответы модели

Смысл: какие точки сложнее или дороже закрывать, где больше откликов.

In [ ]:
#имена колонок могут отличаться — берём из датасета
col_rest = [c for c in df.columns if 'Спецпризнак 1' in c or c == 'Спецпризнак 1'][0] or 'Спецпризнак 1'
col_place = [c for c in df.columns if 'Спецпризнак 2' in c or c == 'Спецпризнак 2'][0] or 'Спецпризнак 2'

rest_metrics = df.groupby(col_rest).agg(
    cost_mean=('Cтоимость отклика, руб.', 'mean'),
    iv_mean=('Индекс вежливости', 'mean'),
    prod_mean=('Откликов на публикацию, шт.', 'mean'),
    responses=('Отклики, шт.', 'sum')
).reset_index()

if col_place in df.columns:
    rest_metrics['Локация'] = df.groupby(col_rest)[col_place].first().values

rest_sorted = rest_metrics.sort_values('cost_mean', ascending=True)

fig = px.bar(
    rest_sorted,
    x=col_rest,
    y='cost_mean',
    color='Локация' if 'Локация' in rest_metrics.columns else None,
    title='Стоимость отклика по подразделениям/точкам',
    labels={'cost_mean': 'Стоимость отклика, руб.', col_rest: ''}
)
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

In [ ]:
rest_metrics['Индекс вежливости'] = pd.to_numeric(rest_metrics['iv_mean'], errors='coerce')
rest_text = rest_metrics.round(2).to_string(index=False)

prompt_rest = f"""Ты HR-аналитик. Объясни менеджеру эффективность подбора по подразделениям/точкам. Говори по-русски, 2–3 абзаца.

ВАЖНО: Данные с HH.ru. Локации и структура — из выгрузки клиента.

СВОДКА:
{rest_text}

Объясни: где подбор дороже, где дешевле, где хуже индекс вежливости и продуктивность, какие выводы."""

response_rest = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_rest}]
)
display(response_rest.choices[0].message.content)

'На основе представленных данных можно проанализировать эффективность подбора персонала по подразделениям и локациям. Например, в Санкт-Петербурге наименьшие затраты на подбор наблюдаются в локации «Ласточка» (44.34), что может говорить о высокой конкурентоспособности данной точки и более привлекательных условиях для соискателей. При этом наибольшие средние затраты у «Гнезда» (54.68), что требует дополнительного внимания к условиям труда и имиджу работодателя в этой локации.\n\nЧто касается индекса вежливости, наивысшие показатели зафиксированы у «Панорамы» (75.64) и «Теремка» (76.73) в Великим Новгороде, что предполагает более качественное взаимодействие с кандидатами и, возможно, лучшую репутацию компании в целом. В то же время, в «Бруснике» (63.64) и «Заре» (70.03) в Санкт-Петербурге наблюдаются более низкие индексы при сравнительно высоком количестве откликов, что может указывать на необходимость улучшения коммуникации с кандидатами.\n\nВ целом, наиболее эффективно подбор осуществл